In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Descargar el modelo y guardarlo en la carpeta 'fidelidade_custom_charts/' con el nombre '5 - Models_2025_09_pets.pkl'
model = "5 - Models_2025_09_pets.pkl"

product = "pets"

labels_order = [
    "non_optimized", "digital-meta", "digital-afiliacion", "digital-google", "digital-billing", "off-television", "off-radio", "off-outdoor", "off-news", "optimized"
]

In [3]:
from meridian.analysis import optimizer
import joblib
import pandas as pd

mmm = joblib.load(model)

In [4]:
# ========================================
# OPTIMIZACIÓN PRESUPUESTARIA 1
# ========================================

start_date = "2025-01-01"   # cambia por la fecha real de inicio en tu data
end_date = "2025-09-01"     # cambia por la fecha real de fin

# Crear el optimizador desde el modelo entrenado
budget_optimizer = optimizer.BudgetOptimizer(mmm)
num_canales = 8

# 1. Definir las restricciones inferiores
# Inicialmente, todos los canales pueden bajar al 70%
lower_constraints = [0.5] * num_canales
lower_constraints[4] = 0

# 2. Definir las restricciones superiores
# Inicialmente, todos los canales pueden subir al 130%
upper_constraints = [0.5] * num_canales

# Ejecutar la optimización con todos los parámetros configurables
optimization_results = budget_optimizer.optimize(
    use_posterior=True,
    selected_times=None,
    fixed_budget=True, # Mantenemos el presupuesto total fijo
    budget=None, # None = usa el gasto histórico total
    start_date = start_date,
    end_date = end_date,
    # Restricciones Ajustadas
    spend_constraint_lower=lower_constraints,
    spend_constraint_upper=upper_constraints,
    target_roi=None,
    target_mroi=None,
    gtol=0.0001,
    use_optimal_frequency=True,
    use_kpi=True,
    confidence_level=0.9,
    batch_size=3000
)


I0000 00:00:1770322980.908239  436285 service.cc:148] XLA service 0x16666ba40 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770322980.908259  436285 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1770322980.914549  436285 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-02-05 15:23:01.682895: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


In [5]:
df_nonopt = optimization_results._get_delta_data(metric='incremental_outcome',
    metric_int="nonopt",
)

In [6]:
# ========================================
# OPTIMIZACIÓN PRESUPUESTARIA 2
# ========================================

start_date = "2025-01-01"   # cambia por la fecha real de inicio en tu data
end_date = "2025-09-01"     # cambia por la fecha real de fin

# Crear el optimizador desde el modelo entrenado
budget_optimizer = optimizer.BudgetOptimizer(mmm)
num_canales = 8

# 1. Definir las restricciones inferiores
# Inicialmente, todos los canales pueden bajar al 70%
lower_constraints = [1] * num_canales

# 2. Definir las restricciones superiores
# Inicialmente, todos los canales pueden subir al 130%
upper_constraints = [1] * num_canales

# Fijamos las reestricciones de cliente aff,,meta, goog,biling,tv,radio,ooh,news
pct_of_spend=[0.11, 0.02, 0.0513, 0, 0.6824, 0, 0.1222, 0.0141]

# Fijamos las reestricciones de cliente
lower_constraints[4] = 0 # Presupuesto de TV no puede bajar
lower_constraints[5] = 0 # Presupuesto de off no puede bajar
lower_constraints[6] = 0 # Presupuesto de off no puede bajar
lower_constraints[7] = 0 # Presupuesto de off no puede bajar

upper_constraints[4] = 0 # Presupuesto de off no puede subir gracias a ese 30%
upper_constraints[5] = 0 # Presupuesto de off no puede subir
upper_constraints[6] = 0 # Presupuesto de off no puede subir gracias a ese 30%
upper_constraints[7] = 0 # Presupuesto de off no puede subir gracias a ese 30%

# Ejecutar la optimización con todos los parámetros configurables
optimization_results = budget_optimizer.optimize(
    use_posterior=True,
    selected_times=None,
    fixed_budget=True, # Mantenemos el presupuesto total fijo
    budget=1372.6, # None = usa el gasto histórico total
    start_date = start_date,
    end_date = end_date,
    # Restricciones Ajustadas
    pct_of_spend=pct_of_spend,
    spend_constraint_lower=lower_constraints,
    spend_constraint_upper=upper_constraints,
    target_roi=None,
    target_mroi=None,
    gtol=0.0001,
    use_optimal_frequency=True,
    use_kpi=True,
    confidence_level=0.9,
    batch_size=3000
)

In [7]:
df_opt = optimization_results._get_delta_data(metric='incremental_outcome',
    metric_int="opt",
)

In [8]:
df_final = pd.merge(
    df_opt, df_nonopt, on='channel', suffixes=('_opt', '_nonopt')
)

# Calcular la diferencia
metric_opt = 'incremental_outcome_opt'
metric_nonopt = 'incremental_outcome_nonopt'
df_final['diff'] = df_final[metric_opt] - df_final[metric_nonopt]

# Convertir en diccionario
data = {item['channel']: round(item['diff'], 6) for item in df_final.to_dict(orient='records')}
data['optimized'] = 0.0
data['non_optimized'] = float(round(df_final[metric_nonopt].sum(), 6))

dict_final = {k: data.get(k, 0.0) for k in labels_order}

In [ ]:
# NOTA: La fuente Space Grotesk debe estar instalada en el sistema para que el gráfico se vea correctamente
output_filename = f"incremental_outcome_delta_{product}.png"
chart = optimization_results.plot_incremental_outcome_delta(df_dict=dict_final, custom_d_e=(10000, 10650))
chart.save(output_filename)
chart

alt.LayerChart(...)